In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Muleta Breeding Program Tutorial - AlphaSimPy

This notebook converts the provided BRAID abstraction for **Muleta** into a tutorial-style AlphaSimPy workflow.
It compares two recurrent selection strategies:

- **PRS**: phenotypic recurrent selection
- **GARS**: genomic-assisted recurrent selection

**Source**: BRAID abstraction loaded from `Muleta.yaml`  
**Package**: AlphaSimPy  
**Style reference**: AlphaSimPy tutorial notebooks in `alphasimpy_tutorials`


## Program Summary

The BRAID abstraction describes a simulated diploid plant breeding program with:

- 10 chromosomes
- 1 additive target trait
- 100 QTL for the target trait
- trait heritability of approximately 0.3
- a simulated founder/base population
- a split into PRS and GARS pathways

The PRS pathway uses phenotype-based recurrent selection with recombination and one generation of selfing.
The GARS pathway uses a training population derived from the base population and recurrent selection on predicted breeding values.


## Assumptions Used in This Notebook

Several quantities were not fully specified in the BRAID diagram, so the notebook uses explicit assumptions to create a coherent runnable AlphaSimPy example:

- Founder population size is set to **100**, matching the BRAID placeholder.
- The base population is expanded by random mating before splitting into PRS and GARS streams.
- The split into PRS and GARS is implemented by selecting two random halves from the same expanded base population.
- Selection intensity of **0.1** is implemented as selecting the top 10% of individuals.
- PRS uses phenotype as the selection criterion after selfing to S1.
- GARS uses a simple RR-BLUP-style genomic prediction helper trained on the base/training population.
- Inbreeding is approximated from heterozygosity because a direct inbreeding helper may not be available in all AlphaSimPy builds.
- The notebook emphasizes clarity and tutorial structure over exact reproduction of every low-level implementation detail in the original diagram.


## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve
from AlphaSimPy import runMacs, SimParam, newPop, randCross, setPheno, selectInd, meanG, varG

print('AlphaSimPy Muleta Tutorial')
print('Libraries imported successfully!')

## Global Parameters

Set the main simulation parameters based on the BRAID abstraction and explicit assumptions.

In [ ]:
# Genome and trait settings from BRAID
n_chr = 10
n_qtl_per_chr = 10   # 10 chromosomes x 10 QTL = 100 total QTL
n_snp_per_chr = 50   # assumption for genomic prediction support
heritability = 0.3

# Founder and population settings
founder_size = 100
base_expand_crosses = 100
base_expand_progeny = 2

# Selection and mating settings from BRAID assumptions
selection_intensity = 0.10
n_crosses = 50
n_progeny = 2

# Cycle horizons from BRAID control loops
n_prs_cycles = 10
n_gars_cycles = 30

# Error variance placeholder from BRAID
var_e = 1.0
reps = 1

print('Simulation Parameters:')
print(f'  Chromosomes: {n_chr}')
print(f'  QTL per chromosome: {n_qtl_per_chr}')
print(f'  SNP per chromosome: {n_snp_per_chr}')
print(f'  Founder size: {founder_size}')
print(f'  PRS cycles: {n_prs_cycles}')
print(f'  GARS cycles: {n_gars_cycles}')
print(f'  Selection intensity: {selection_intensity}')
print(f'  Crosses per cycle: {n_crosses}')

## Helper Functions

The genomic-assisted pathway needs helper functions for subsetting populations, extracting SNP genotypes,
fitting a simple RR-BLUP-style model, and assigning EBVs.

In [ ]:
def subsetPop(pop, indices):
    from AlphaSimPy import Pop
    indices = list(indices)
    if len(indices) == 0:
        return Pop(
            n_ind=0, n_chr=pop.n_chr, ploidy=pop.ploidy, n_loci=pop.n_loci,
            geno=[], gen_map=pop.gen_map, centromere=pop.centromere,
            inbred=pop.inbred, id=[], iid=[], mother=[], father=[], sex=[],
            n_traits=pop.n_traits, gv=np.empty((0, pop.n_traits)),
            pheno=np.empty((0, pop.n_traits)), ebv=np.empty((0, 0)),
            gxe=pop.gxe, fix_eff=[], misc={}, misc_pop={}
        )
    return Pop(
        n_ind=len(indices), n_chr=pop.n_chr, ploidy=pop.ploidy, n_loci=pop.n_loci,
        geno=[pop.geno[chr_idx][:, :, indices] for chr_idx in range(pop.n_chr)],
        gen_map=pop.gen_map, centromere=pop.centromere, inbred=pop.inbred,
        id=[pop.id[i] for i in indices],
        iid=[pop.iid[i] for i in indices],
        mother=[pop.mother[i] for i in indices],
        father=[pop.father[i] for i in indices],
        sex=[pop.sex[i] for i in indices],
        n_traits=pop.n_traits,
        gv=pop.gv[indices, :],
        pheno=pop.pheno[indices, :],
        ebv=pop.ebv[indices, :] if hasattr(pop, 'ebv') and pop.ebv is not None and pop.ebv.size > 0 else np.empty((len(indices), 0)),
        gxe=pop.gxe,
        fix_eff=[pop.fix_eff[i] for i in indices],
        misc=pop.misc,
        misc_pop=pop.misc_pop
    )


def pullSnpGeno(pop, simParam, snpChip=1):
    snp_chip = simParam.snp_chips[snpChip - 1]
    n_snp = sum(snp_chip.loci_per_chr)
    geno_matrix = np.zeros((pop.n_ind, n_snp), dtype=float)
    snp_idx = 0
    for chr_idx in range(pop.n_chr):
        n_snp_chr = snp_chip.loci_per_chr[chr_idx]
        if n_snp_chr == 0:
            continue
        snp_loci = snp_chip.loci_loc[snp_idx:snp_idx + n_snp_chr]
        for ind_idx in range(pop.n_ind):
            for local_idx, snp_loc in enumerate(snp_loci):
                byte_idx = snp_loc // 8
                bit_idx = snp_loc % 8
                genotype = 0
                for p in range(pop.ploidy):
                    byte_val = pop.geno[chr_idx][byte_idx, p, ind_idx]
                    genotype += (byte_val >> bit_idx) & 1
                geno_matrix[ind_idx, snp_idx + local_idx] = genotype
        snp_idx += n_snp_chr
    return geno_matrix


def RRBLUP(trainPop, simParam, traits=1, use='pheno', snpChip=1):
    if use == 'pheno':
        y = trainPop.pheno[:, traits - 1]
    elif use == 'gv':
        y = trainPop.gv[:, traits - 1]
    else:
        raise ValueError('use must be pheno or gv in this notebook')
    valid_idx = ~np.isnan(y)
    y = y[valid_idx]
    M = pullSnpGeno(trainPop, simParam, snpChip)
    M = M[valid_idx, :]
    M_mean = np.mean(M, axis=0)
    M_centered = M - M_mean
    n_markers = M_centered.shape[1]
    G = np.dot(M_centered, M_centered.T) / max(n_markers, 1)
    G += np.eye(G.shape[0]) * 1e-6
    lambda_val = max(n_markers / 10.0, 1.0)
    u = solve(G + lambda_val * np.eye(G.shape[0]), y, assume_a='pos')
    return {
        'u': u,
        'M_mean': M_mean,
        'M_train': M_centered,
        'trait': traits,
        'lambda': lambda_val
    }


def setEBV(pop, gsModel, simParam, snpChip=1):
    M = pullSnpGeno(pop, simParam, snpChip)
    M_centered = M - gsModel['M_mean']
    K = np.dot(M_centered, gsModel['M_train'].T) / max(gsModel['M_train'].shape[1], 1)
    ebv = np.dot(K, gsModel['u']).reshape(-1, 1)
    pop.ebv = ebv
    return pop


def selectTopFraction(pop, frac, use='pheno', simParam=None):
    n_select = max(1, int(np.ceil(pop.n_ind * frac)))
    return selectInd(pop, n_select, use=use, simParam=simParam)


def selfPop(pop, simParam, nProgeny=1):
    return randCross(pop, pop.n_ind, nProgeny, parents=1, simParam=simParam)


def approxInbreeding(pop, simParam, snpChip=1):
    try:
        M = pullSnpGeno(pop, simParam, snpChip)
        p = np.mean(M / 2.0, axis=0)
        het_obs = np.mean(M == 1)
        het_exp = np.mean(2 * p * (1 - p))
        if het_exp <= 0:
            return np.nan
        return 1.0 - (het_obs / het_exp)
    except Exception:
        return np.nan

print('Helper functions defined successfully!')

## Simulate Founders and Create the Base Population

This section creates the founder haplotypes, initializes the simulation parameters, and expands the base population.
The expanded base population is then used both as a training set and as the source for the PRS and GARS pathways.

In [ ]:
# Simulate founder haplotypes
founderPop = runMacs(nInd=founder_size, nChr=n_chr, segSites=200, inbred=False)

# Set simulation parameters
SP = SimParam(founderPop)
SP.addTraitA(nQtlPerChr=n_qtl_per_chr)
SP.setVarE(h2=heritability)
SP.addSnpChip(nSnpPerChr=n_snp_per_chr)

# Create founder/base population
base_population = newPop(founderPop, simParam=SP)
base_population = setPheno(base_population, varE=var_e, reps=reps, simParam=SP)

# Expand base population by one generation of random mating
base_population = randCross(base_population, base_expand_crosses, base_expand_progeny, simParam=SP)
base_population = setPheno(base_population, varE=var_e, reps=reps, simParam=SP)

print('Base population created')
print(f'  Individuals in base population: {base_population.n_ind}')
print(f'  Mean genetic value: {meanG(base_population)[0]:.3f}')
print(f'  Genetic variance: {varG(base_population)[0]:.3f}')

## Split the Base Population into PRS and GARS Streams

The BRAID abstraction splits the base population into two pathways. Here, the split is implemented as a random partition.

In [ ]:
rng = np.random.default_rng(2024)
all_idx = np.arange(base_population.n_ind)
rng.shuffle(all_idx)
half = base_population.n_ind // 2

prs_idx = all_idx[:half]
gars_idx = all_idx[half:]

c0_prs = subsetPop(base_population, prs_idx)
c0_gars = subsetPop(base_population, gars_idx)

# Training population for genomic prediction
training_pop = base_population

print(f'PRS C0 size: {c0_prs.n_ind}')
print(f'GARS C0 size: {c0_gars.n_ind}')
print(f'Training population size: {training_pop.n_ind}')

## Run the PRS and GARS Recurrent Selection Pipelines

This section follows the BRAID workflow:

- **PRS**: select on phenotype, recombine, self to S1, phenotype, and reselect.
- **GARS**: train a genomic prediction model, predict EBVs, select on EBV, and recombine.

Results are stored each cycle for comparison.

In [ ]:
results_prs = []
results_gars = []

# ---- PRS initialization ----
c0_prs = setPheno(c0_prs, varE=var_e, reps=reps, simParam=SP)
prs_selected = selectTopFraction(c0_prs, selection_intensity, use='pheno', simParam=SP)

results_prs.append({
    'cycle': 0,
    'pathway': 'PRS',
    'meanG': meanG(prs_selected)[0],
    'varG': varG(prs_selected)[0],
    'inbreeding': approxInbreeding(prs_selected, SP)
})

for cycle in range(1, n_prs_cycles + 1):
    prs_recombined = randCross(prs_selected, n_crosses, n_progeny, simParam=SP)
    prs_s1 = selfPop(prs_recombined, SP, nProgeny=1)
    prs_s1 = setPheno(prs_s1, varE=var_e, reps=reps, simParam=SP)
    prs_selected = selectTopFraction(prs_s1, selection_intensity, use='pheno', simParam=SP)
    results_prs.append({
        'cycle': cycle,
        'pathway': 'PRS',
        'meanG': meanG(prs_selected)[0],
        'varG': varG(prs_selected)[0],
        'inbreeding': approxInbreeding(prs_selected, SP)
    })

print('PRS pipeline completed')

# ---- GARS initialization ----
c0_gars = setPheno(c0_gars, varE=var_e, reps=reps, simParam=SP)
gars_selected = selectTopFraction(c0_gars, selection_intensity, use='pheno', simParam=SP)
gars_s0 = randCross(gars_selected, n_crosses, n_progeny, simParam=SP)

# Initial model from base/training population
training_pop = setPheno(training_pop, varE=var_e, reps=reps, simParam=SP)
gsModel = RRBLUP(training_pop, SP, traits=1, use='pheno', snpChip=1)

gars_s0 = setEBV(gars_s0, gsModel, SP, snpChip=1)
gars_selected = selectTopFraction(gars_s0, selection_intensity, use='ebv', simParam=SP)

results_gars.append({
    'cycle': 0,
    'pathway': 'GARS',
    'meanG': meanG(gars_selected)[0],
    'varG': varG(gars_selected)[0],
    'inbreeding': approxInbreeding(gars_selected, SP)
})

for cycle in range(1, n_gars_cycles + 1):
    gsModel = RRBLUP(training_pop, SP, traits=1, use='pheno', snpChip=1)
    gars_evaluated = setEBV(gars_s0, gsModel, SP, snpChip=1)
    gars_selected = selectTopFraction(gars_evaluated, selection_intensity, use='ebv', simParam=SP)
    results_gars.append({
        'cycle': cycle,
        'pathway': 'GARS',
        'meanG': meanG(gars_selected)[0],
        'varG': varG(gars_selected)[0],
        'inbreeding': approxInbreeding(gars_selected, SP)
    })
    gars_s0 = randCross(gars_selected, n_crosses, n_progeny, simParam=SP)

print('GARS pipeline completed')

## Combine and Inspect Results

Convert the stored outputs into data frames for easier inspection.

In [ ]:
prs_df = pd.DataFrame(results_prs)
gars_df = pd.DataFrame(results_gars)

print('PRS results head:')
display(prs_df.head())

print('GARS results head:')
display(gars_df.head())

## Plot Genetic Mean, Genetic Variance, and Inbreeding

The BRAID abstraction requested tracking of genetic mean, genetic variance, and inbreeding.
The plots below compare the two pathways over their respective cycle horizons.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 14))

axes[0].plot(prs_df['cycle'], prs_df['meanG'], marker='o', label='PRS')
axes[0].plot(gars_df['cycle'], gars_df['meanG'], marker='s', label='GARS')
axes[0].set_title('Genetic Mean by Cycle', fontweight='bold')
axes[0].set_xlabel('Cycle')
axes[0].set_ylabel('Mean Genetic Value')
axes[0].grid(True, linestyle=':')
axes[0].legend()

axes[1].plot(prs_df['cycle'], prs_df['varG'], marker='o', label='PRS')
axes[1].plot(gars_df['cycle'], gars_df['varG'], marker='s', label='GARS')
axes[1].set_title('Genetic Variance by Cycle', fontweight='bold')
axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Genetic Variance')
axes[1].grid(True, linestyle=':')
axes[1].legend()

axes[2].plot(prs_df['cycle'], prs_df['inbreeding'], marker='o', label='PRS')
axes[2].plot(gars_df['cycle'], gars_df['inbreeding'], marker='s', label='GARS')
axes[2].set_title('Approximate Inbreeding by Cycle', fontweight='bold')
axes[2].set_xlabel('Cycle')
axes[2].set_ylabel('Approximate Inbreeding')
axes[2].grid(True, linestyle=':')
axes[2].legend()

plt.tight_layout()
plt.show()

## Summary Tables

Summarize the initial and final outcomes for each pathway.

In [ ]:
summary_df = pd.DataFrame([
    {
        'pathway': 'PRS',
        'initial_meanG': prs_df['meanG'].iloc[0],
        'final_meanG': prs_df['meanG'].iloc[-1],
        'initial_varG': prs_df['varG'].iloc[0],
        'final_varG': prs_df['varG'].iloc[-1],
        'final_inbreeding': prs_df['inbreeding'].iloc[-1]
    },
    {
        'pathway': 'GARS',
        'initial_meanG': gars_df['meanG'].iloc[0],
        'final_meanG': gars_df['meanG'].iloc[-1],
        'initial_varG': gars_df['varG'].iloc[0],
        'final_varG': gars_df['varG'].iloc[-1],
        'final_inbreeding': gars_df['inbreeding'].iloc[-1]
    }
])

summary_df['genetic_gain'] = summary_df['final_meanG'] - summary_df['initial_meanG']
display(summary_df)

print('Completed Muleta BRAID-to-AlphaSimPy tutorial notebook.')

## Notes

This notebook is a faithful **tutorial-style translation** of the BRAID abstraction rather than a claim of exact biological calibration.
If more detailed program parameters become available, the following parts can be refined further:

- chromosome lengths and recombination map
- exact founder haplotype simulation settings
- exact population sizes at each stage
- genomic prediction model specification
- explicit inbreeding calculation from pedigree or genomic relationship matrices
